# Bank Campaign Conversion Prediction
## Binary classification using Logistic Regression, LDA and QDA

**Dataset:** UCI Bank Marketing (`bank-full.csv`, 45,211 rows)  
**Target:** `y` — whether the customer subscribed to a term deposit (`yes` / `no`)  
**Prediction timing:** Before the marketing call. Therefore, `duration` is excluded to prevent target leakage.

### Project objectives
1. Explore and prepare the bank marketing data.
2. Compare a majority-class baseline with Logistic Regression, Linear Discriminant Analysis (LDA), and Quadratic Discriminant Analysis (QDA).
3. Use stratified train/test splitting and preprocessing without leakage.
4. Evaluate precision, recall, F1-score, ROC-AUC and confusion matrices.
5. Perform feature selection and interpret Logistic Regression coefficients.
6. Discuss model-based targeting versus a simple baseline.

> Run the notebook from top to bottom. In Google Colab, upload the supplied `bank+marketing.zip` when prompted.


## 1. Imports and reproducibility

In [ ]:
import io, zipfile, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report,
                             RocCurveDisplay, ConfusionMatrixDisplay)
from sklearn.feature_selection import SelectFromModel

RANDOM_STATE = 42
TEST_SIZE = 0.20
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')


## 2. Load the dataset

In [ ]:
# Option A (Google Colab): upload the original bank+marketing.zip.
# Option B (local Jupyter): set ZIP_PATH to the location of your ZIP file.
try:
    from google.colab import files
    uploaded = files.upload()
    outer_zip_bytes = next(iter(uploaded.values()))
except Exception:
    ZIP_PATH = 'bank+marketing.zip'  # Change this path if needed
    with open(ZIP_PATH, 'rb') as f:
        outer_zip_bytes = f.read()

outer = zipfile.ZipFile(io.BytesIO(outer_zip_bytes))
print('Files in outer ZIP:', outer.namelist())

# The supplied archive contains bank.zip, which contains bank-full.csv.
inner_zip_bytes = outer.read('bank.zip')
inner = zipfile.ZipFile(io.BytesIO(inner_zip_bytes))
csv_name = 'bank-full.csv'
df = pd.read_csv(inner.open(csv_name), sep=';')

print('Dataset shape:', df.shape)
display(df.head())


## 3. Data understanding

In [ ]:
print('Shape:', df.shape)
print('\\nData types:')
display(df.dtypes.to_frame('dtype'))
print('\\nSummary statistics:')
display(df.describe(include='all').T)
print('\\nMissing values by column:')
display(df.isna().sum().sort_values(ascending=False).to_frame('missing'))
print('Duplicate rows:', df.duplicated().sum())
print('\\nTarget counts:')
display(df['y'].value_counts().to_frame('count'))
print('\\nTarget proportions:')
display(df['y'].value_counts(normalize=True).mul(100).round(2).to_frame('percent'))


## 4. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=df, x='y')
plt.title('Term deposit subscription (target distribution)')
plt.xlabel('Subscribed?'); plt.ylabel('Number of customers')
plt.show()

plt.figure(figsize=(9,5))
sns.histplot(data=df, x='age', hue='y', bins=30, stat='density',
             common_norm=False, element='step')
plt.title('Age distribution by subscription outcome')
plt.show()

plt.figure(figsize=(10,5))
sns.countplot(data=df, x='job', hue='y')
plt.xticks(rotation=45, ha='right')
plt.title('Subscription outcome by job')
plt.tight_layout(); plt.show()

plt.figure(figsize=(10,5))
sns.boxplot(data=df, x='y', y='balance')
plt.title('Account balance by subscription outcome')
plt.show()


### Initial observations
- The target is imbalanced: most customers did not subscribe.
- Accuracy alone can be misleading; the majority-class classifier can achieve high accuracy while finding no subscribers.
- `duration` is excluded because it is only known after the call. Including it would leak information unavailable at prediction time.
- `pdays = -1` is a dataset code meaning the customer was not previously contacted; it should not be treated as an ordinary negative number. We retain the original coding for this baseline comparison and can engineer a `previously_contacted` flag in an extension.


## 5. Preprocessing and train/test split

In [ ]:
# Drop target and duration. The model predicts before a call takes place.
X = df.drop(columns=['y', 'duration'])
y = df['y'].map({'no': 0, 'yes': 1})

categorical_features = X.select_dtypes(include=['object']).columns.tolist()
numeric_features = X.select_dtypes(exclude=['object']).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

numeric_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer([
    ('num', numeric_pipe, numeric_features),
    ('cat', categorical_pipe, categorical_features)
], verbose_feature_names_out=False)

# Fit preprocessing only on training data to avoid data leakage.
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)
feature_names = preprocessor.get_feature_names_out()

print('Training rows:', X_train.shape[0])
print('Testing rows:', X_test.shape[0])
print('Processed feature count:', len(feature_names))
print('Positive rate in train:', round(y_train.mean(), 4))
print('Positive rate in test:', round(y_test.mean(), 4))


## 6. Train models

In [ ]:
models = {
    'Majority baseline': DummyClassifier(strategy='most_frequent'),
    'Logistic Regression': LogisticRegression(
        max_iter=2000, class_weight='balanced', solver='lbfgs',
        random_state=RANDOM_STATE
    ),
    'LDA': LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto'),
    'QDA': QuadraticDiscriminantAnalysis(reg_param=0.10)
}

fitted_models = {}
rows = []
predictions = {}
probabilities = {}

for name, model in models.items():
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        model.fit(X_train_processed, y_train)
        y_pred = model.predict(X_test_processed)
        fitted_models[name] = model
        predictions[name] = y_pred
        if hasattr(model, 'predict_proba'):
            y_prob = model.predict_proba(X_test_processed)[:, 1]
            probabilities[name] = y_prob
            auc = roc_auc_score(y_test, y_prob)
        else:
            probabilities[name] = None
            auc = np.nan

    rows.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1-score': f1_score(y_test, y_pred, zero_division=0),
        'ROC-AUC': auc
    })

results = pd.DataFrame(rows).set_index('Model').sort_values('F1-score', ascending=False)
display(results.round(4))


## 7. Model comparison visualizations

In [ ]:
results[['Accuracy','Precision','Recall','F1-score','ROC-AUC']].plot(
    kind='bar', figsize=(12,6), ylim=(0,1), rot=20
)
plt.title('Model performance comparison')
plt.ylabel('Score')
plt.tight_layout(); plt.show()

plt.figure(figsize=(7,6))
for name, prob in probabilities.items():
    if prob is not None:
        RocCurveDisplay.from_predictions(y_test, prob, name=name, ax=plt.gca())
plt.plot([0,1], [0,1], linestyle='--', label='Random classifier')
plt.title('ROC curves')
plt.legend(); plt.show()


## 8. Confusion matrices and classification reports

In [ ]:
for name, y_pred in predictions.items():
    print('\\n', name)
    print(classification_report(y_test, y_pred, target_names=['No','Yes'], zero_division=0))
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred, display_labels=['No','Yes'], cmap='Blues', values_format='d'
    )
    plt.title(f'Confusion matrix — {name}')
    plt.show()


## 9. Feature selection and coefficient interpretation

Use L1-regularized Logistic Regression as a feature-selection method. The selector is fit only on the training data. The selected columns are then used to fit a second Logistic Regression model.

For a positive coefficient, increasing that encoded feature increases the model's log-odds of subscription, holding other features constant. For a negative coefficient, it decreases the log-odds. Coefficients are not causal effects.


In [ ]:
selector_model = LogisticRegression(
    penalty='l1', solver='saga', C=0.15, max_iter=3000,
    class_weight='balanced', random_state=RANDOM_STATE
)
selector_model.fit(X_train_processed, y_train)
selector = SelectFromModel(selector_model, prefit=True, threshold='median')
selected_mask = selector.get_support()
selected_features = feature_names[selected_mask]

print('Selected features:', len(selected_features), 'of', len(feature_names))
print(selected_features.tolist())

Xtr_selected = selector.transform(X_train_processed)
Xte_selected = selector.transform(X_test_processed)
selected_lr = LogisticRegression(
    max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE
)
selected_lr.fit(Xtr_selected, y_train)
selected_pred = selected_lr.predict(Xte_selected)
selected_prob = selected_lr.predict_proba(Xte_selected)[:,1]

print(classification_report(y_test, selected_pred, target_names=['No','Yes']))
print('Selected Logistic Regression F1:', round(f1_score(y_test, selected_pred),4))
print('Selected Logistic Regression ROC-AUC:', round(roc_auc_score(y_test, selected_prob),4))

coef_table = pd.DataFrame({
    'feature': selected_features,
    'coefficient': selected_lr.coef_[0]
}).sort_values('coefficient')
display(pd.concat([coef_table.head(10), coef_table.tail(10)]))


## 10. Model-based targeting versus a simple baseline

In [ ]:
# Baseline: contact nobody selectively; majority-class prediction labels every customer 'No'.
# Model-based targeting: rank customers by predicted subscription probability.
# The following table compares the top 10% highest-scored test customers.
best_model_name = results['F1-score'].idxmax()
best_model = fitted_models[best_model_name]
best_prob = probabilities[best_model_name]

if best_prob is not None:
    ranked = pd.DataFrame({'actual': y_test.to_numpy(), 'score': best_prob})
    ranked = ranked.sort_values('score', ascending=False)
    top_n = max(1, int(0.10 * len(ranked)))
    top_group = ranked.head(top_n)
    print('Best F1 model:', best_model_name)
    print('Top 10% customers:', top_n)
    print('Subscription rate in top 10%:', round(top_group['actual'].mean(),4))
    print('Subscription rate in full test set:', round(ranked['actual'].mean(),4))
    print('Lift over test-set average:', round(top_group['actual'].mean()/ranked['actual'].mean(),2))


## 11. Conclusion

Use the printed test-set metrics to identify the best model for the business objective. Do not select a model using accuracy alone. If the bank wants to find as many likely subscribers as possible, recall matters; if call-center capacity is limited, precision and lift in the highest-ranked segment matter. The decision threshold should be tuned using validation data and business costs, not the test set.

### Limitations and future improvements
- The dataset is observational and model associations are not causal.
- Results depend on the random split and preprocessing choices.
- Tune classification thresholds and hyperparameters with cross-validation.
- Test class-weighting, SMOTE (inside training folds only), and calibration.
- Compare the `bank-additional-full.csv` dataset separately if economic indicators are required.
- Monitor performance and fairness before deploying a campaign model.
